# 🛡️ LangChain Middleware: Message-Based Summarization

Middleware ek aisa intermediate layer hai jo agent aur language model ke beech mein request aur response ko intercept, inspect, aur modify karta hai.

### Why SummarizationMiddleware?
- Jab conversation lambi ho jaati hai, toh message history LLM ke token limit ko exceed kar sakti hai.
- **`SummarizationMiddleware`** message history monitor karta hai aur threshold reach hone par purane messages ko automatically summarize kar deta hai taaki context preserve rahe aur context overflow na ho.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

### 🤖 Model Initialization
- **Purpose**: Groq provider ke zariye `openai/gpt-oss-20b` chat model initialize karna.
- **Parameters**:
  - `model`: Model name string (`'openai/gpt-oss-20b'`).
  - `model_provider`: Provider name (`'groq'`).
- **Return**: Initialized LLM instance ready for agent integration.

In [ ]:
llm = init_chat_model('openai/gpt-oss-20b', model_provider='groq')

### 🧠 Message-Based Summarization Agent
- **Class / Function**: `create_agent` with `SummarizationMiddleware`
- **Purpose**: Ek stateful agent create karna jo 10 messages ke baad purani chat history ko summarize kare aur context window ko optimize rakhe.
- **Parameters**:
  - `model`: Base LLM jo agent use karega.
  - `checkpointer`: `MemorySaver()` jo session state aur conversation history persist karta hai.
  - `middleware`: Middleware list containing `SummarizationMiddleware`.
  - `trigger`: `('messages', 10)` — Threshold specifying that summarization triggers after 10 messages.
  - `keep`: `('messages', 4)` — Summarization ke baad 4 most recent messages ko verbatim (as it is) preserve karta hai.

In [ ]:
# Message-based summarization agent
agent = create_agent(
    model=llm,
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=('messages', 10),
            keep=('messages', 4)
        )
    ]
)

agent

### 💬 Testing Stateful Execution with Thread ID
- **Purpose**: Agent ko conversation thread ke sath invoke karna taaki memory aur summarization test ho sake.
- **Parameters**:
  - `messages`: List of user messages (`HumanMessage`).
  - `config`: Dictionary specifying `thread_id` for state tracking.

In [ ]:
config = {'configurable': {'thread_id': 'conversation_1'}}

response = agent.invoke(
    {'messages': [HumanMessage(content='Hello! My name is Mohit Payasi.')]},
    config=config
)

print(response['messages'][-1].content)